# Job Criteria

Goal: Extract detailed job requirements for downstream LLM scoring of candidates.

In [ ]:
import json
from pathlib import Path

from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache
from langchain_openai import ChatOpenAI

from recruit.pydantic import RoleRequirements

# Set global cache to in-memory
set_llm_cache(InMemoryCache())

In [ ]:
llm = ChatOpenAI(
    model="gpt-5.6-terra",
    temperature=0.0,
)
# llm = ChatAnthropic(model="claude-opus-5")  # type: ignore

In [ ]:
requirements_agent = llm.with_structured_output(
    RoleRequirements,
    method="json_schema",
)

In [ ]:
openenings = Path("..") / "openings"
# job_description_path = openenings / "frontend-engineer-berlin.md"
job_description_path = openenings / "technical-founders-associate-berlin.md"
# job_description_path = openenings / "ai-success-engineer-berlin.md"
# job_description_path = openenings / "president-and-coo-london-berlin.md"

with open(job_description_path, "r") as file:
    job_description = file.read()

In [ ]:
system_prompt = f"""
Extract assessment criteria from the job description that can be objectively evaluated from a candidate's resume or profile.

Scope of Criteria:
- Technical Skills & Tools (languages, frameworks, platforms)
- Work Experience & Seniority (years in role, track record, leadership, company stage/scale)
- Education & Domain Background (degrees, fields of study, relevant industry experience)

Guidelines:
- Ensure all criteria are mutually exclusive and collectively cover the key aspects of the role.
- Focus strictly on verifiable attributes observable from a CV/profile (avoid generic soft skills like "curious" or "team player").
- Required: Non-negotiable, mandatory criteria (limit to 3 core criteria).
- Preferred: Strongly advantageous, nice-to-have qualifications (limit to 5-8 criteria).
""".strip()

In [ ]:
messages = [
    ("system", system_prompt),
    ("human", job_description),
]

raw = requirements_agent.invoke(messages)
requirements = RoleRequirements.model_validate(raw)

In [ ]:
print(requirements.model_dump_json(indent=2))

In [ ]:
with open("../spi/requirements.json", "w") as f:
    json.dump(requirements.model_dump(), f, indent=2)